# Load Data into SQL, SQL Cleaning, Export clean data, and EDA

In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv('../data/raw/online_retail_II.csv', encoding='ISO-8859-1')

print(df.shape) # check total rows and columns
print(df.columns.tolist()) # printed all column names
print(df.dtypes) # data type of all columns

(1067371, 8)
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object


In [3]:
conn = sqlite3.connect('../data/retail.db')
df.to_sql('raw_transactions', conn, if_exists='replace', index=False) 

# confirm it loaded
check = pd.read_sql('SELECT COUNT(*) as total_rows FROM raw_transactions', conn)
check

,total_rows
0,1067371


SQL CLEANING

In [4]:
# 1. Nulls per column
nulls = pd.read_sql('''
SELECT 
  SUM(CASE WHEN Invoice IS NULL THEN 1 ELSE 0 END) AS null_invoice,
  SUM(CASE WHEN StockCode IS NULL THEN 1 ELSE 0 END) AS null_stockcode,
  SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description,
  SUM(CASE WHEN Quantity IS NULL THEN 1 ELSE 0 END) AS null_quantity,
  SUM(CASE WHEN InvoiceDate IS NULL THEN 1 ELSE 0 END) AS null_invoicedate,
  SUM(CASE WHEN Price IS NULL THEN 1 ELSE 0 END) AS null_price,
  SUM(CASE WHEN "Customer ID" IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
  SUM(CASE WHEN Country IS NULL THEN 1 ELSE 0 END) AS null_country
FROM raw_transactions
''', conn)
print(nulls)

   null_invoice  null_stockcode  null_description  null_quantity  \
0             0               0              4382              0   

   null_invoicedate  null_price  null_customer_id  null_country  
0                 0           0            243007             0  


In [5]:
# 2. Full duplicate rows
dupes = pd.read_sql('''
SELECT COUNT(*) as duplicate_rows
FROM (
    SELECT Invoice, StockCode, Description, Quantity, InvoiceDate, Price, "Customer ID", Country, COUNT(*) as cnt
    FROM raw_transactions
    GROUP BY Invoice, StockCode, Description, Quantity, InvoiceDate, Price, "Customer ID", Country
    HAVING cnt > 1
)
''', conn)
print(dupes)

   duplicate_rows
0           32907


In [6]:
# 3. Cancellations (Invoice starting with 'C')
cancellations = pd.read_sql('''
SELECT COUNT(*) as cancelled_rows
FROM raw_transactions
WHERE Invoice LIKE 'C%'
''', conn)
print(cancellations)

   cancelled_rows
0           19494


In [7]:
# 4. Negative or zero Quantity/Price (excluding cancellations, since those are expected to be negative)
bad_values = pd.read_sql('''
SELECT
    SUM(CASE WHEN Quantity <= 0 AND INVOICE NOT LIKE 'C%' THEN 1 ELSE 0 END) AS negative_qty_not_cancelled,
    SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS zero_or_negative_price
FROM raw_transactions
''', conn)
print(bad_values)

   negative_qty_not_cancelled  zero_or_negative_price
0                        3457                    6207


In [8]:
# checking negative qty not cancelled
anomaly_check = pd.read_sql('''
SELECT Invoice, StockCode, Description, Quantity, Price, "Customer ID"
FROM raw_transactions
WHERE Quantity <= 0 AND Invoice NOT LIKE 'C%'
LIMIT 20
''', conn)
print(anomaly_check)

   Invoice StockCode      Description  Quantity  Price Customer ID
0   489464     21733     85123a mixed       -96    0.0        None
1   489463     71477            short      -240    0.0        None
2   489467    85123A      21733 mixed      -192    0.0        None
3   489521     21646             None       -50    0.0        None
4   489655     20683             None       -44    0.0        None
5   489660     35956             lost     -1043    0.0        None
6   489663    35605A          damages      -117    0.0        None
7   489806     18010             None      -770    0.0        None
8   489820     21133  invcd as 84879?      -720    0.0        None
9   489821    85049G             None      -240    0.0        None
10  489899   79323GR     sold as gold      -954    0.0        None
11  489901     21098             None      -200    0.0        None
12  490007     84347            21494      -720    0.0        None
13  490016     21982             None     -1012    0.0        

In [9]:
# checking the negative or zero price
price_check = pd.read_sql('''
SELECT Invoice, StockCode, Description, Quantity, Price
FROM raw_transactions
WHERE price <= 0
LIMIT 20
''', conn)
print(price_check)

   Invoice StockCode          Description  Quantity  Price
0   489464     21733         85123a mixed       -96    0.0
1   489463     71477                short      -240    0.0
2   489467    85123A          21733 mixed      -192    0.0
3   489521     21646                 None       -50    0.0
4   489655     20683                 None       -44    0.0
5   489659     21350                 None       230    0.0
6   489660     35956                 lost     -1043    0.0
7   489663    35605A              damages      -117    0.0
8   489781     84292                 None        17    0.0
9   489806     18010                 None      -770    0.0
10  489820     21133      invcd as 84879?      -720    0.0
11  489821    85049G                 None      -240    0.0
12  489825     22076   6 RIBBONS EMPIRE          12    0.0
13  489861       DOT       DOTCOM POSTAGE         1    0.0
14  489882    35751C                 None        12    0.0
15  489898    79323G                 None       954    0

In [10]:
# check if the table cleaned_transaction already created and only clean record
result = pd.read_sql('SELECT COUNT(*) as cleaned_rows FROM cleaned_transactions', conn)
print(result)

   cleaned_rows
0        779425


FEATURE ENGINEERING - make derived columns for EDA and rfm analysis

In [11]:
cleaned = pd.read_sql('SELECT * FROM cleaned_transactions', conn)
cleaned

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
779420,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
779421,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
779422,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
779423,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [12]:
cleaned = pd.read_sql('SELECT * FROM cleaned_transactions', conn)

# Convert InvoiceDate data type to datetime
cleaned['InvoiceDate'] = pd.to_datetime(cleaned['InvoiceDate'])

# Extract year and month
cleaned['Year'] = cleaned['InvoiceDate'].dt.year
cleaned['Month'] = cleaned['InvoiceDate'].dt.month

# Line-level revenue
cleaned['LineRevenue'] = cleaned['Quantity'] * cleaned ['Price']

# Quick check
print(cleaned[['InvoiceDate', 'Year', 'Month', 'Quantity', 'Price', 'LineRevenue']])
print(cleaned.dtypes)

               InvoiceDate  Year  Month  Quantity  Price  LineRevenue
0      2009-12-01 07:45:00  2009     12        12   6.95        83.40
1      2009-12-01 07:45:00  2009     12        12   6.75        81.00
2      2009-12-01 07:45:00  2009     12        12   6.75        81.00
3      2009-12-01 07:45:00  2009     12        48   2.10       100.80
4      2009-12-01 07:45:00  2009     12        24   1.25        30.00
...                    ...   ...    ...       ...    ...          ...
779420 2011-12-09 12:50:00  2011     12         6   2.10        12.60
779421 2011-12-09 12:50:00  2011     12         4   4.15        16.60
779422 2011-12-09 12:50:00  2011     12         4   4.15        16.60
779423 2011-12-09 12:50:00  2011     12         3   4.95        14.85
779424 2011-12-09 12:50:00  2011     12         1  18.00        18.00

[779425 rows x 6 columns]
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate  

In [13]:
## convert new cleaned dataset to csv
cleaned.to_csv('../data/cleaned/retail_cleaned.csv', index=False)

## EDA Questions

1. Monthly revenue trend
2. Top 10 Products by Revenue
3. Top 10 Product by Quantity
4. Revenue by country

In [18]:
cleaned.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year,Month,LineRevenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009,12,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009,12,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009,12,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009,12,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009,12,30.0


In [15]:
# 1. monthly revenue trend
monthly_revenue = cleaned.groupby(cleaned['InvoiceDate'].dt.to_period('M'))['LineRevenue'].sum()
print(monthly_revenue)

InvoiceDate
2009-12     683504.010
2010-01     555802.672
2010-02     504558.956
2010-03     696978.471
2010-04     591982.002
2010-05     597833.380
2010-06     636371.130
2010-07     589736.170
2010-08     602224.600
2010-09     829013.951
2010-10    1033112.010
2010-11    1166460.022
2010-12     570422.730
2011-01     568101.310
2011-02     446084.920
2011-03     594081.760
2011-04     468374.331
2011-05     677355.150
2011-06     660046.050
2011-07     598962.901
2011-08     644051.040
2011-09     950690.202
2011-10    1035642.450
2011-11    1156205.610
2011-12     517208.440
Freq: M, Name: LineRevenue, dtype: float64


In [ ]:
# 2. Top 10 products by revenue 
top_products_revenue = cleaned.groupby('Description')['LineRevenue'].sum().sort_values(ascending=False).head(10)
print(top_products_revenue)

# findings: manual and postage aren't real products

Description
REGENCY CAKESTAND 3 TIER              277656.25
WHITE HANGING HEART T-LIGHT HOLDER    247048.01
PAPER CRAFT , LITTLE BIRDIE           168469.60
Manual                                151777.67
JUMBO BAG RED RETROSPOT               134307.44
POSTAGE                               124648.04
ASSORTED COLOUR BIRD ORNAMENT         124351.86
PARTY BUNTING                         103283.38
MEDIUM CERAMIC TOP STORAGE JAR         81416.73
PAPER CHAIN KIT 50'S CHRISTMAS         76598.18
Name: LineRevenue, dtype: float64


In [ ]:

# check unique non-standard codes (usually short/alphabetic codes = not real products)
cleaned['StockCode'].value_counts().head(20)

# findings: sort by frequency count not quite right here, cause there could be other non-product codes with lower counts

StockCode
85123A    5023
22423     3335
85099B    3296
84879     2692
20725     2609
21212     2557
47566     2098
20727     2045
22383     2039
21034     1950
21232     1935
22382     1935
22384     1874
21754     1852
22139     1848
20914     1821
22469     1821
20728     1820
84991     1813
POST      1803
Name: count, dtype: int64

In [ ]:
# searching for non-numeric cause real product always start with numeric or numeric+letter,instead using value count
non_standard = cleaned[cleaned['StockCode'].str.match(r'^\d+[A-Za-z]?$') == False]['StockCode'].value_counts()
print(non_standard)

# findings: possible non product: post, m, c2, adjust, bank charges, dot, test001, d, adjust2, and test002

StockCode
POST            1803
15056BL          799
M                681
C2               248
79323LP          159
79323GR           76
ADJUST            32
BANK CHARGES      31
PADS              17
DOT               16
TEST001            9
D                  5
ADJUST2            3
SP1002             2
TEST002            1
Name: count, dtype: int64


In [ ]:
# need to check possible product by seeing the description
check_codes = ['15056BL', '79323LP', '79323GR', 'PADS', 'SP1002']
for code in check_codes:
    print(code, '->', cleaned[cleaned['StockCode'] == code]['Description'].unique())

# findings: all the possible product is a real product

15056BL -> ['EDWARDIAN PARASOL BLACK']
79323LP -> ['LIGHT PINK CHERRY LIGHTS']
79323GR -> ['GREEN CHERRY LIGHTS']
PADS -> ['PADS TO MATCH ALL CUSHIONS']
SP1002 -> ["KID'S CHALKBOARD/EASEL"]


In [34]:
non_product_codes = ['POST', 'M', 'C2', 'ADJUST', 'ADJUST2', 'BANK CHARGES', 'DOT', 'TEST001', 'TEST002', 'D']
products_only = cleaned[~cleaned['StockCode'].isin(non_product_codes)]

# re-run top products, now excluding non-product entries
top_products_revenue = products_only.groupby('Description')['LineRevenue'].sum().sort_values(ascending=False).head(10)
print('Top 10 Products by Revenue\n', top_products_revenue)
top_products_qty = products_only.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
print('\n\nTop 10 Products by Quantity\n', top_products_qty)

Top 10 Products by Revenue
 Description
REGENCY CAKESTAND 3 TIER              277656.25
WHITE HANGING HEART T-LIGHT HOLDER    247048.01
PAPER CRAFT , LITTLE BIRDIE           168469.60
JUMBO BAG RED RETROSPOT               134307.44
ASSORTED COLOUR BIRD ORNAMENT         124351.86
PARTY BUNTING                         103283.38
MEDIUM CERAMIC TOP STORAGE JAR         81416.73
PAPER CHAIN KIT 50'S CHRISTMAS         76598.18
CHILLI LIGHTS                          69084.30
JUMBO BAG STRAWBERRY                   64127.77
Name: LineRevenue, dtype: float64


Top 10 Products by Quantity
 Description
WORLD WAR 2 GLIDERS ASSTD DESIGNS     105185
WHITE HANGING HEART T-LIGHT HOLDER     91757
PAPER CRAFT , LITTLE BIRDIE            80995
ASSORTED COLOUR BIRD ORNAMENT          78234
MEDIUM CERAMIC TOP STORAGE JAR         77916
JUMBO BAG RED RETROSPOT                74224
BROCADE RING PURSE                     70082
PACK OF 60 PINK PAISLEY CAKE CASES     54592
60 TEATIME FAIRY CAKE CASES            5282

In [22]:
# 3. Top 10 products by quantity
top_products_qty = cleaned.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
print(top_products_qty)

Description
WORLD WAR 2 GLIDERS ASSTD DESIGNS     105185
WHITE HANGING HEART T-LIGHT HOLDER     91757
PAPER CRAFT , LITTLE BIRDIE            80995
ASSORTED COLOUR BIRD ORNAMENT          78234
MEDIUM CERAMIC TOP STORAGE JAR         77916
JUMBO BAG RED RETROSPOT                74224
BROCADE RING PURSE                     70082
PACK OF 60 PINK PAISLEY CAKE CASES     54592
60 TEATIME FAIRY CAKE CASES            52828
PACK OF 72 RETRO SPOT CAKE CASES       45129
Name: Quantity, dtype: int64


In [23]:
# 4. Revenue by country
revenue_by_country = cleaned.groupby('Country')['LineRevenue'].sum().sort_values(ascending=False)
print(revenue_by_country)

Country
United Kingdom          1.438923e+07
EIRE                    6.165705e+05
Netherlands             5.540381e+05
Germany                 4.250197e+05
France                  3.487690e+05
Australia               1.692835e+05
Spain                   1.083325e+05
Switzerland             1.000619e+05
Sweden                  9.151582e+04
Denmark                 6.858069e+04
Belgium                 6.538782e+04
Norway                  5.632250e+04
Portugal                5.555478e+04
Channel Islands         4.462333e+04
Japan                   4.302391e+04
Italy                   3.210817e+04
Finland                 2.992554e+04
Singapore               2.531706e+04
Cyprus                  2.484995e+04
Austria                 2.361301e+04
Greece                  1.909619e+04
Poland                  1.065429e+04
Israel                  1.041524e+04
United Arab Emirates    9.202690e+03
Unspecified             8.607350e+03
USA                     8.366860e+03
Malta                   8.0990